# Sprint 007 D0 — Investigation readiness

Read-only validation of accepted Sprint 006 artifacts. This notebook calls `sprint007_artifact_validation` only — no bridge math or granular economics interpretation.

In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "backtest").is_dir() and (candidate / "setup.py").exists():
            return candidate
    raise RuntimeError("Could not locate MomentumCVG repo root (set cwd or PYTHONPATH)")


REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.backtest.sprint007_artifact_validation import (
    OFFICIAL_RUN_DIR,
    run_d0_validation,
    write_manifest,
)

result = run_d0_validation()
print("Provisional verdict:", result.verdict)
for gate in result.gates:
    status = "PASS" if gate.passed else "FAIL"
    print(f"{gate.gate_id} [{status}] {gate.detail}")

## Question-to-field sufficiency (accepted calculation)

| Sprint need | Artifacts | D0 confirmation |
|---|---|---|
| D1 midpoint margin | `trade_log_mid`, `date_summary_mid` | Primary P&L/CAR fields present |
| D2 dollar bridge | paired `trade_log_*`, `leg_log_*` | Trade + leg quote/fill fields present; pairing gates passed |
| D3 execution requirement | paired legs + trade cost ratios | Leg `fill_price` on `leg_log`; ratios on `trade_log` |
| Package fill probability | — | **Unanswerable** from EOD snapshots |
| Alternative structures | — | **Unanswerable** without new run |
| Pure Momentum/CVG IC | funnel/candidate only | **Out of scope** |

In [ ]:
EVIDENCE_DIR = Path(
    f"C:/MomentumCVG_env/runs/sprint007_d0_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = write_manifest(result, EVIDENCE_DIR / "d0_artifact_manifest.json")
manifest_path